GENERAZIONE DI TESTO CON GPT-2: DALL'NIDIREZIONALITA' ALLA CREATIVITA'

Oltre a BERT, l'arltra grande famiglia dei Transformer è GPT

BERT legge il testo in entrambe le direzioni, utilizzato soprattutto per comprensione/rappresentazione
GPT-2 guarda solo i token precedenti, predice il token successivo, utilizzato per generazione di testo.
GPT-2 è quindi un Transformer decoder-only autoregressivo

Partiamo dalla frase: "The cas is sitting on the"
GPT-2 deve prevedere quale token potrebbe venire dopo:
mat, chair, floor, table, ...
Assegna una probabilità a moltissimo possibili token 
mat (0.35), chair (0.20), floor (0.15), table (0.08), ...
ne sceglie uno, esempio mat
e ora il testo diventa
"The cat is sitting on the mat"
Poi ripete:
"The cat is sitting on the mat" prossimo token?
magari genera "and"
quindi 
"The cat is sitting on the mat and" e continua 
Questo è il significato di AUTOREGRESSIVO.
Ogni nuovo token viene generato utilizzando anche i token che il modello ha già generato.

La pipeline è:
prompt -> GPT-2 -> token successivo -> aggiunto il token al testo -> GPT-2 di nuovo -> token successivo -> ...

Perchè si parla di uniderizionalità?
GPT-2 usa self-attention, quindi non è una RNN che fisicamente trasporta uno stato da sinistra verso destra. Però applica una causal mask che impedisce a ogni posizione di vedere i token futuri.
Supponiamo la  frase:
"A love deep learning"
Durante l'addestramento:
I - può vedere I
love - può vedere I love
deep - può vedere I love deep
learning - può vedere I love deep learning
ma quando deve prevedere learning, non può già vedere learning perchè la maschera lo impedisce (perchè durante la generazione reale il futuro non esiste ancora)
Lo può già vedere, ma la maschera lo impedisce.
Durante il training il sistema riconosce anche la parola "learning" e la usa come target, cioè confronta la sua risposta (previsione) con la risposta corretta. Da qui calcola la loss ed i pesi vengono aggiornati/ottimizzati nel caso sia errata la risposta.

Quind la differenza con BERT, può guardare sia a destra che sinistra. BIDIREZIONALE
GPT invece può guardare solo quanto c'è a sinistra. AUTOREGRESSIVO (con la maschera)

Ed è proprio questa caratteristica che rende GPT naturalmente adatto alla generazione.

Con questo meccanismo GPT impara: sequenze di parole, grammatica, relazioni tra concetti, stili, regolarità del linguaggio

Ma cosa entra realmente nel modello? 
Come con BERT:
testo -> tokenizer -> token -> token IDs -> embedding -> Transformer

La differenza rispetto a BERT è nell'attenzione
GPT-2 Masked Self-Attention - nessun token può vedere il futuro.

Mentra BERT alla fine deve prevedere positivo/negativo quindi:
BERT - [CLS] 768 valori - Dense - positivo/negativo
GPT-2 deve decidere quale dei token nel vocabolario deve venire dopo

Pertanto, per tutto il vocabolario, deve produrre un punteggio
Questo valori vengono chiamati logits
Supponiamo, per assurdo, un vocabolario di 5 parole (cat, dog, house, car, mat)
GPT potrebbe produrre
cat - 1.2
dog - 1.4
house - -0.8
car - -1.1
mat - 2.7
i logits vengono trasformati in probabilità
cat - 0.51
dog - 0.016
house - 0.02
car - 0.01
mat - 0.76
e il modello sceglie mat
Nella realtà però il vocabolario contiene decine di migliaia di token

Ed eccoci alla Creatività
Il modello non deve sempre scegliere il token con probabilità maggiore
Nell'esempio precedente se sceglie sempre 'mat' il testo tende a diventare molto deterministico
Ma puoi cambiare (con dei parametri) questa impostazioe e magari fargli scegliere car (anche se aveva  una probabilità inferiore)

I parametri che cambiano questa impostazione negli LLM sono:
    - temperature: è il parametro più intuitivo. Temperatura bassa (es 0.2) rende la distribuzione più concentrata sui token probabili, quindi è più prevedibile, conservativo, meno variabile. Temperatura alta (es 1.2) rende più probabile scegliere anche le alternative meno ovvie, quini più varietà, più creatività, ma anche maggior rischio di testo assurdo.
    - top_k: con top_k=50 dici "considera solamente i 50 token più probabili e scarta tutti gli altri". Quindi, invece di scegliere tra tutto il vocabolario (es 50.000 token considera solo i 50 token più probabili e scarta tutti gli altir)
    - top_p: è più dinamico rispetto a top_k, top_p=0.9, considera i token più probabili finchè la loro probabilità cumulativa raggiunge circa il 90%. Questo taglia la code lunga di token improbabili, questo previene errori imbarazzanti dove il modello inserisce parole completamente fuori contesto.
    - Greedy Search: la strategia base che sceglie sempre il token con probabilità massima, spesso portando a loop ripetitivi

Trovare l'equilibrio tra coerenza e creatività è un arte
Una temperatura troppo bassa rende il modello molto conservativo e ripetivo, mentre una temperatura alta, aumenta il rischio di allucinazioni e testi sconnessi.
A differenza di Top-k, top-p si adatta alla forma della distribuzione: se molti token sono plausibili, il set si espande, se uno è quasi certo, il set di restringe.
La Ripetition Penalty e la tecnica completamentare che penalizza i token già presenti nella sequenza per favorire la diversità lessicale.

La temperatura T interviene dividendo i logit z prima dell'applicazione della funzione softmax. Questo altera la confidenza del modello senza cambiare l'ordine dei token
Per T che tende a zero, la parola con il punteggio più alto schiaccia tutte le altre, rendendo la scelta deterministica (Greedy); per T elevato, le differenze tra le parole si appiattiscono, la distribuzione tende a diventare uniforme, rendendo quasi equiparabile la scelta di una parola sensata e di un termine casuale.

Come dirle alla macchina cosa deve dire
Prompt Engineering: Guidare il Modello
La struttura dell'input condizionale.
GPT-2 non ha una volonta propria, è un sistema puramente reattivo, l'output ottenuto è lo specchio dell'input fornito. Progettare un prompot non è solo scrivere una domanda, ma costruire un contesto condizionale. Più dettagliato e coerente è il prompt più GPT riesce a rimanere nell'argomento.
GPT-2 genera testo basandosi esclusivamente sulla sequenza di input fornita. La qualità e la forma del prompot determinano drasticamente il comportamento dell'output
La progettazione di un prompt permette di simulare diversi stili, toni, formati e compiti logici senza modificare il pesi del modello.

Tecniche principali per istruire GPT
Dell'input al risultato desiderato
Esistono diversi modi per dialogare con GPT
- Zero-shot Prompting: fornire un istruzione diretta al modello senza esempi (es. 'traduci questa frase in inflese:')
- Few-Shot Prompting: includere alcuni esempi di input-output nel prompt per aiutare il modello a catturare i pattern richiesto
- Contesto di Sistema: definire un ruolo o una personalità per il modello all'interno della sequenza iniziale. Esempio puoi dire al modello "comportati come un assistente tecnico"
- Delimitatori e Stop Tokens: utilizzare caratteri speciali per separare le istruzioni dal testo e definire dove la generazione deve fermarsi.

Ma come rendiamo questi prompt davvero efficaci?
Ottimizzazione del Prompt è un processo empirico, quasi scientifico
- Chiarezza e Specificità: Prompt vaghi producono risultati generici. Essere specifici sul formato desiderato (es. Scrivi un elenco puntato) riduce l'incertezza del modello
- Iterazione del Prompt: il processo è spesso empirico: piccoli cambiamenti nella punteggiatura o nell'ordine delle parole possono variare sensibilmente l'output
- Lunghezza del Contesto: è necessario monitorare che il prompt non occupi troppa memoria, lasciando spazio sufficiente alla generazione dei token richiesti. GPT ha una memoria limitata, chiamata finestra di contesto, se il prompot è troppo lungo il modello non ha abbastanza spazio per generare una risposta articolata.

Implementazione con Huggin Face
AutoModelForCausalLM e pipeline generativa
Per lavorare con GPT-2 usiamo Hugging Face, la classe regina è AutoModelForCausalLM, causal è la parola chiave, indica modelli che vedono solo il passato, perfetti per la generazione.
Caricare il modello è quindi un opearzione di una riga, ma configurare l'inferenza richiede attenzione per i parametri.

- Carichiamo il corpo del transformer e la testa lineare di output per la predizione dei token, con AutoModelForCausalL;
- Metodo generate(): la funzione principale che incapsula la logica di campionamento e la gestione della memoria
- Pad Token ID: configurazione necessaria per gestire sequenze di lunghezza diversa durante l'elaborazione di batch. Specialmente se lavoriamo con più frasi contemporaneamente.
- Device Mapping: spostamento del modello su GPU per accelerare il processo di generazione di token. La generazione di token è un calcolo iterativo e lo spostamento su  GPU ne beneficia.

Parametri di generate()
- max_new_tokens: definisce il limite superiore di parole da generare, prevedendo l'esaurimento delle risorse computazionali
- do_sample: boolean fondamentale: se impostato a False, il modello utilizzerà la ricerca Greedy ignorando la temperature e top-p, se true attiviamo la creatività
- Early-stopping: interrompe la generazione non appena tutti i flussi in batch raggiungono un token di fine sequenza (EOS). il modello smette di parlare non appena chiuso il concetto.

La Catena di GEnerazione
La generazione è una catena ed ogni parola generata non è solo visualizzata ma reintrodotto nel modello come se facesse parte del prompt originale.
La probabilità dell'intera storia, è il prodotto delle probabilità di ogni singolo passo.
Un errore all'inizio può essere portato fino alla fine.

BERT - "Ti do tutto il testo: dimmi che cosa significa / rappresentalo."
GPT - "Ti do il testo fino a questo punto: dimmi cosa potrebbe venire dopo."    


In [1]:
"""
Controllo Totale della Generazione GPT-2
-----------------------------------------------------------
Questo script carica GPT-2 e genera testo utilizzando un unico prompt,
configurando e spiegando ogni parametro di campionamento direttamente.
"""

import os

# Configurazione obbligatoria per usare Keras con l'anima di PyTorch
os.environ["KERAS_BACKEND"] = "torch"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def esempio_generazione_semplice():
    """
    Esegue una singola generazione di testo commentando ogni parametro tecnico.
    """
    
    # 1. PREPARAZIONE (Caricamento rapido)
    print("Caricamento modello...")
    model_id = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    
    # Spostamento su GPU se presente per velocizzare (Best Practice)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # 2. DEFINIZIONE DEL PROMPT (Input)
    prompt = "In the year 2050, artificial intelligence will"
    
    # Trasformiamo il testo in numeri (Input IDs)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 3. GENERAZIONE CON TUTTI I PARAMETRI (Il cuore della lezione)
    print(f"\nPrompt iniziale: {prompt}")
    print("-" * 30)

    output_tokens = model.generate(
        **inputs,
        
        # --- Parametri di Lunghezza ---
        max_new_tokens=40,       # Quanti nuovi token (parole) generare al massimo
        
        # --- Cuore del Sampling ---
        do_sample=True,          # TRUE: abilita la fantasia (sampling). FALSE: usa la Greedy Search (sempre il più probabile)
        
        # --- Strategie di Strategia ---
        temperature=0.8,         # Controlla la confidenza: <1.0 = conservativo, >1.0 = creativo/caotico
        top_k=50,                # Limita la scelta alle 50 parole più probabili (riduce il rischio di errori gravi)
        top_p=0.92,              # Nucleus Sampling: sceglie tra le parole che sommate arrivano al 92% di probabilità
        
        # --- Gestione della Ripetizione ---
        repetition_penalty=1.2,  # Evita che il modello scriva la stessa parola o frase all'infinito
        
        # --- Configurazione Tecnica ---
        pad_token_id=tokenizer.eos_token_id  # Indica al modello come gestire gli spazi vuoti
    )

    # 4. TRADUZIONE OUTPUT (Da numeri a parole)
    testo_generato = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    
    print(f"Risultato:\n{testo_generato}")

if __name__ == "__main__":
    esempio_generazione_semplice()

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Caricamento modello...


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3101.51it/s]



Prompt iniziale: In the year 2050, artificial intelligence will
------------------------------
Risultato:
In the year 2050, artificial intelligence will replace more than 2 million jobs in US healthcare and medicine.
/ AFP 2018 / SITE HIGHLIGHTS 'THRILLED BY THE FUTURE' UPI - AUSTRAL
